## Cell 1: Environment Setup & Library Imports

In [ ]:
!pip install nibabel scikit-learn
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob, re, time
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit, train_test_split

## Cell 2: User Configuration & Path Definitions

In [ ]:
# ======== USER CONFIGURATION ========
# Set DATASETS_DIR to the folder in your Google Drive containing all datasets.
DATASETS_DIR = '/content/drive/MyDrive/datasets'  # Update if using a different directory
# ====================================

AFRICA_BASE        = os.path.join(DATASETS_DIR, 'BraTS-Africa')
OUTPUT_ROOT        = os.path.join(DATASETS_DIR, 'braTS_africa_test', 'glioma')
SLICES_PER_PATIENT = 5
IMG_SIZE           = (224, 224)

def process_all_african_scans():
    # Slice extraction is pre-computed; confirms the 95-patient / 475-slice expectation.
    print('T1c slice extraction for the 95 Glioma patients (475 total slices) is already complete.')

process_all_african_scans()

## Cell 3: Local SSD Cache Setup & Data Verification

In [ ]:
# Clear and rebuild local SSD cache to prevent stale state
if os.path.exists('/tmp/datasets'):
    shutil.rmtree('/tmp/datasets')
    print('Cleared stale local cache.')

print('Copying dataset from Drive to local SSD...')
shutil.copytree(DATASETS_DIR, '/tmp/datasets')

# Verify Nigerian healthy scans (25 axial slices from 5 subjects)
healthy_path = '/tmp/datasets/nigerian_healthy/axial'
files = (glob.glob(os.path.join(healthy_path, '**', '*.png'), recursive=True) +
         glob.glob(os.path.join(healthy_path, '**', '*.jpg'), recursive=True))
print(f'Found {len(files)} healthy image files in local SSD.')

## Cell 4: Hardware & Hardware Acceleration Check

In [ ]:
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Cell 5: Dataset Classes, Data Transformations & Directory Parsing

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEEDS       = [12, 23, 42, 2026]
SPLIT_RATIO = 0.40
NUM_EPOCHS  = 25
BATCH_SIZE  = 32
NUM_WORKERS = 2
USE_AMP     = device.type == 'cuda'

LOCAL_BASE = '/tmp/datasets'
if not os.path.exists(LOCAL_BASE):
    print('Copying dataset from Drive to local SSD...')
    shutil.copytree(DATASETS_DIR, LOCAL_BASE)

BASE_DIR            = LOCAL_BASE
AFRICAN_TUMOR_DIR   = os.path.join(BASE_DIR, 'braTS_africa_test', 'glioma')
WESTERN_TUMOR_DIR   = os.path.join(BASE_DIR, 'tumor')
WESTERN_HEALTHY_DIR = os.path.join(BASE_DIR, 'notumor')
AFRICAN_HEALTHY_DIR = os.path.join(BASE_DIR, 'nigerian_healthy', 'axial')

SAVE_WEIGHTS_DIR = os.path.join(os.path.dirname(DATASETS_DIR), 'checkpoints')
os.makedirs(SAVE_WEIGHTS_DIR, exist_ok=True)


class MRIDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def extract_patient_id(filename):
    base = os.path.basename(filename)
    return base.split('_slice')[0] if '_slice' in base else base.split('.')[0]


def scan_dir(directory, label, filter_duplicates=False):
     # Returns a DataFrame of image paths, labels, and patient IDs for a directory.
    filepaths, labels, groups = [], [], []
    if os.path.exists(directory):
        for ext in ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG'):
            for f in sorted(glob.glob(os.path.join(directory, '**', ext), recursive=True)):
                filename = os.path.basename(f)
                 # Exclude Drive sync duplicates (e.g. 'file (1).png') when requested
                if filter_duplicates and (re.search(r'\(\d+\)', filename) or 'copy' in filename.lower()):
                    continue
                filepaths.append(f)
                labels.append(label)
                groups.append(extract_patient_id(f))

    df = pd.DataFrame({'filepath': filepaths, 'label': labels, 'patient_id': groups})
    return df.sort_values(by=['patient_id', 'filepath']).reset_index(drop=True)


df_afr_tumor    = scan_dir(AFRICAN_TUMOR_DIR,   label=1)
df_west_tumor   = scan_dir(WESTERN_TUMOR_DIR,   label=1)
df_west_healthy = scan_dir(WESTERN_HEALTHY_DIR, label=0, filter_duplicates=True)
df_afr_healthy  = scan_dir(AFRICAN_HEALTHY_DIR, label=0)

print('Setup complete.')
print(f'  African tumour images:   {len(df_afr_tumor)}')
print(f'  African healthy images:  {len(df_afr_healthy)}')
print(f'  Western tumour images:   {len(df_west_tumor)}')
print(f'  Western healthy images:  {len(df_west_healthy)}')

## Cell 6: DINOv2 Feature Caching & Multi-Seed Training

In [ ]:
num_cpus = os.cpu_count() or 4
torch.set_num_threads(num_cpus)
print(f'Configured PyTorch for CPU execution using {num_cpus} threads.')

device = torch.device('cpu')

# Head-only classifier — takes cached 384-dim backbone features as input.
# This redefines DINOv2Classifier for the feature-cached training path.
class DINOv2Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(384, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.classifier(x).squeeze(-1)


preprocess_transform = val_transforms

print('Loading DINOv2 backbone on CPU...')
start_time = time.time()
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
backbone.eval()


def cache_features(df, transform, dataset_name='Dataset'):
    if df is None or len(df) == 0:
        print(f'Warning: {dataset_name} is empty.')
        return torch.empty((0, 384)), torch.empty((0,))

    dataset = MRIDataset(df['filepath'].values, df['label'].values, transform)
    loader  = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)

    all_features, all_labels = [], []
    total_batches = len(loader)
    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            feats = backbone(images.to(device)).cpu()
            all_features.append(feats)
            all_labels.append(labels)
            if (i + 1) % max(1, total_batches // 4) == 0 or (i + 1) == total_batches:
                print(f'   [{dataset_name}] batch {i+1}/{total_batches}')

    return torch.cat(all_features, dim=0), torch.cat(all_labels, dim=0)


if len(df_west_tumor) >= 1800:
    df_west_tumor_train   = df_west_tumor.iloc[:1400].copy()
    df_west_tumor_val     = df_west_tumor.iloc[1400:1800].copy()
    df_west_healthy_train = df_west_healthy.iloc[:1400].copy()
    df_west_healthy_val   = df_west_healthy.iloc[1400:1800].copy()
else:
    df_west_tumor_train,   df_west_tumor_val   = train_test_split(df_west_tumor,   test_size=0.2, random_state=42)
    df_west_healthy_train, df_west_healthy_val = train_test_split(df_west_healthy, test_size=0.2, random_state=42)

print('\nCaching feature representations...')
feats_afr_tumor, labels_afr_tumor = cache_features(df_afr_tumor, preprocess_transform, 'African Tumour')
feats_afr_healthy, labels_afr_healthy = cache_features(df_afr_healthy, preprocess_transform, 'Nigerian Healthy')

feats_west_tumor_train,   labels_west_tumor_train   = cache_features(df_west_tumor_train,   preprocess_transform, 'Western Tumour Train')
feats_west_healthy_train, labels_west_healthy_train = cache_features(df_west_healthy_train, preprocess_transform, 'Western Healthy Train')
feats_west_tumor_val,     labels_west_tumor_val     = cache_features(df_west_tumor_val,     preprocess_transform, 'Western Tumour Val')
feats_west_healthy_val,   labels_west_healthy_val   = cache_features(df_west_healthy_val,   preprocess_transform, 'Western Healthy Val')

val_feats  = torch.cat([feats_west_tumor_val,  feats_west_healthy_val],  dim=0)
val_labels = torch.cat([labels_west_tumor_val, labels_west_healthy_val], dim=0)
val_loader = DataLoader(TensorDataset(val_feats, val_labels), batch_size=128, shuffle=False)

elapsed = time.time() - start_time
print(f'Feature extraction complete in {elapsed:.1f} s.')

# Multi-seed training loop across {12, 23, 42, 2026}
print('\nTraining classification head across 4 seeds...')
train_start = time.time()

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    # 60/40 patient-level split on BraTS-Africa (285 train slices / 190 holdout)
    gss = GroupShuffleSplit(n_splits=1, test_size=SPLIT_RATIO, random_state=seed)
    train_idx, _ = next(gss.split(df_afr_tumor, groups=df_afr_tumor['patient_id']))

    afr_train_feats  = feats_afr_tumor[train_idx]
    afr_train_labels = labels_afr_tumor[train_idx]

    train_feats  = torch.cat([afr_train_feats, feats_west_tumor_train, feats_west_healthy_train], dim=0)
    train_labels = torch.cat([afr_train_labels, labels_west_tumor_train, labels_west_healthy_train], dim=0)

    perm = torch.randperm(len(train_feats))
    train_feats, train_labels = train_feats[perm], train_labels[perm]

    train_loader = DataLoader(TensorDataset(train_feats, train_labels), batch_size=BATCH_SIZE, shuffle=True)

    model     = DINOv2Classifier().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

    PATIENCE         = 7
    patience_counter = 0
    best_val_loss    = float('inf')
    best_weights     = None

    for epoch in range(NUM_EPOCHS):
        model.train()
        for features, labels in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for features, labels in val_loader:
                val_loss += criterion(model(features), labels).item() * features.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
            best_weights     = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    if best_weights is not None:
        torch.save(best_weights, os.path.join(SAVE_WEIGHTS_DIR, f'dinov2_seed_{seed}.pth'))

    print(f'   Seed {seed} saved (best val loss: {best_val_loss:.4f})')

print(f'Training complete in {time.time() - train_start:.1f} s.')

## Cell 7: Multi-Seed FP32 Evaluation (Table I)

In [ ]:
device_cpu = torch.device('cpu')
SEEDS      = [12, 23, 42, 2026]


def print_formatted_table(title, df):
    headers = list(df.columns)
    data    = df.values.tolist()
    widths  = [
        max(len(str(h)), max(len(str(row[i])) for row in data))
        for i, h in enumerate(headers)
    ]
    border     = '─' * (sum(widths) + 3 * (len(widths) - 1))
    header_str = '   '.join(
        f'{headers[i]:<{widths[i]}}' if i == 0 else f'{headers[i]:>{widths[i]}}'
        for i in range(len(headers))
    )
    print(title)
    print(border)
    print(header_str)
    print(border)
    for row in data:
        print('   '.join(
            f'{str(row[i]):<{widths[i]}}' if i == 0 else f'{str(row[i]):>{widths[i]}}'
            for i in range(len(row))
        ))
    print(border)


# Nigerian healthy DataLoader (25 slices, 5 subjects)
healthy_loader = DataLoader(
    MRIDataset(df_afr_healthy['filepath'].values, df_afr_healthy['label'].values, val_transforms),
    batch_size=32, shuffle=False
)


# =============================================================
# PART 1: MULTI-SEED EVALUATION — TABLE I
# =============================================================
print('Evaluating checkpoints across seeds...\n')
results = []

for seed in SEEDS:
    # 40% patient-level holdout (38 patients / 190 slices)
    gss = GroupShuffleSplit(n_splits=1, test_size=SPLIT_RATIO, random_state=seed)
    _, test_idx = next(gss.split(df_afr_tumor, groups=df_afr_tumor['patient_id']))
    df_afr_holdout_seed = df_afr_tumor.iloc[test_idx].reset_index(drop=True)

    holdout_loader_seed = DataLoader(
        MRIDataset(df_afr_holdout_seed['filepath'].values, df_afr_holdout_seed['label'].values, val_transforms),
        batch_size=32, shuffle=False
    )

    backbone_eval = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device_cpu)
    backbone_eval.eval()
    for p in backbone_eval.parameters():
        p.requires_grad = False

    head_eval = DINOv2Classifier().to(device_cpu)
    head_eval.load_state_dict(torch.load(
        os.path.join(SAVE_WEIGHTS_DIR, f'dinov2_seed_{seed}.pth'), map_location=device_cpu
    ))
    head_eval.eval()

    # Western validation metrics
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for features, labels in val_loader:
            probs = head_eval(features.to(device_cpu))
            val_loss    += nn.BCELoss()(probs, labels.to(device_cpu)).item() * features.size(0)
            val_correct += (probs >= 0.5).float().eq(labels.to(device_cpu)).sum().item()
    val_loss /= len(val_loader.dataset)
    val_acc   = val_correct / len(val_loader.dataset) * 100.0

    # External sensitivity (BraTS-Africa holdout — 190 slices)
    sens_correct = 0
    with torch.no_grad():
        for imgs, lbls in holdout_loader_seed:
            feats = backbone_eval(imgs.to(device_cpu))
            preds = (head_eval(feats) >= 0.5).float()
            sens_correct += preds.eq(lbls.to(device_cpu)).sum().item()
    sensitivity = sens_correct / len(holdout_loader_seed.dataset) * 100.0

    # External specificity (Nigerian healthy — 25 slices)
    spec_correct = 0
    with torch.no_grad():
        for imgs, lbls in healthy_loader:
            feats = backbone_eval(imgs.to(device_cpu))
            preds = (head_eval(feats) >= 0.5).float()
            spec_correct += ((1 - preds) == (1 - lbls.to(device_cpu))).sum().item()
    specificity = spec_correct / len(healthy_loader.dataset) * 100.0

    results.append({'Seed': seed, 'Val Acc (%)': val_acc, 'Val Loss': val_loss,
                    'Sens. (%)': sensitivity, 'Spec. (%)': specificity})

df_results = pd.DataFrame(results)
mean_row   = {'Seed': 'Mean', **{c: df_results[c].mean() for c in df_results.columns if c != 'Seed'}}
sd_row     = {'Seed': 'SD',   **{c: df_results[c].std()  for c in df_results.columns if c != 'Seed'}}
df_table1  = pd.concat([df_results, pd.DataFrame([mean_row, sd_row])], ignore_index=True)

df_table1_fmt = pd.DataFrame({
    'Seed':        df_table1['Seed'].astype(str),
    'Val Acc (%)': df_table1['Val Acc (%)'].map(lambda x: f'{x:.2f}'),
    'Val Loss':    df_table1['Val Loss'].map(lambda x: f'{x:.4f}'),
    'Sens. (%)':   df_table1['Sens. (%)'].map(lambda x: f'{x:.2f}'),
    'Spec. (%)':   df_table1['Spec. (%)'].map(lambda x: f'{x:.2f}'),
})
print_formatted_table('TABLE I.  Per-Seed Performance (Mean ± SD across Four Seeds)', df_table1_fmt)

## Cell 8: Figure 1 Generation (Probability Distribution Plotting)

In [ ]:
plt.rcParams.update({
    'font.size': 16, 'axes.labelsize': 18, 'axes.titlesize': 18,
    'xtick.labelsize': 16, 'ytick.labelsize': 16
})

fig, axes = plt.subplots(4, 2, figsize=(15, 18), sharex=True)
bin_edges  = np.linspace(0.0, 1.0, 16)

for i, seed in enumerate(SEEDS):
    gss = GroupShuffleSplit(n_splits=1, test_size=SPLIT_RATIO, random_state=seed)
    _, test_idx = next(gss.split(df_afr_tumor, groups=df_afr_tumor['patient_id']))

    holdout_feats = feats_afr_tumor[test_idx]

    model = DINOv2Classifier().to(device)
    model.load_state_dict(torch.load(
        os.path.join(SAVE_WEIGHTS_DIR, f'dinov2_seed_{seed}.pth'), map_location=device
    ))
    model.eval()

    with torch.no_grad():
        brats_probs   = model(holdout_feats.to(device)).cpu().numpy()
        healthy_probs = model(feats_afr_healthy.to(device)).cpu().numpy()

    sens = (brats_probs   >= 0.5).mean() * 100.0
    spec = (healthy_probs  < 0.5).mean() * 100.0

    ax_l = axes[i, 0]
    ax_l.hist(brats_probs, bins=bin_edges, range=(0.0, 1.0),
              color='#1b7837', edgecolor='black', alpha=0.85, linewidth=1.2)
    ax_l.axvline(0.5, color='red', linestyle='--', linewidth=2.0)
    ax_l.set_title(f'Seed {seed} - BraTS holdout (sens {sens:.2f}%)', fontweight='bold', fontsize=18, pad=12)
    ax_l.set_ylabel('Count', fontweight='bold', fontsize=18)
    ax_l.set_xlim(-0.02, 1.02)
    ax_l.tick_params(axis='both', labelsize=16)
    ax_l.grid(True, linestyle=':', alpha=0.4)

    ax_r = axes[i, 1]
    ax_r.hist(healthy_probs, bins=bin_edges, range=(0.0, 1.0),
              color='#4575b4', edgecolor='black', alpha=0.85, linewidth=1.2)
    ax_r.axvline(0.5, color='red', linestyle='--', linewidth=2.0)
    ax_r.set_title(f'Seed {seed} - Nigerian healthy (spec {spec:.2f}%)', fontweight='bold', fontsize=18, pad=12)
    ax_r.set_ylabel('Count', fontweight='bold', fontsize=18)
    ax_r.set_xlim(-0.02, 1.02)
    ax_r.tick_params(axis='both', labelsize=16)
    ax_r.grid(True, linestyle=':', alpha=0.4)

    if i == 3:
        ax_l.set_xlabel('Tumour probability', fontweight='bold', fontsize=18)
        ax_r.set_xlabel('Tumour probability', fontweight='bold', fontsize=18)

plt.tight_layout()
fig_path = os.path.join(SAVE_WEIGHTS_DIR, 'figure1_probability_distributions.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()